In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys

root_path = os.path.abspath("../..")
sys.path.append(root_path)

from algorithm.MADDPG_old import MADDPGTrainer, MADDPGTester
from network_env.network_env_v8 import NetworkEnvV8

### Directories

In [ ]:
RESOURCE_PATH = "./configs/resource_config.json"
LOG_PATH = "./results"
DATA = "./data"

### Configurations

In [ ]:

frames_per_batch = 100
n_iter = 500
min_replay_size = 1000
memory_size = 10000
n_optimizer_steps = 100
train_batch_size = 128
actor_lr = 1e-4
critic_lr = 1e-4
max_grad_norm = 0.5
gamma = 0.99
polyak_tau = 0.005

critic_configs = {
        "num_cells":256,
        "depth":2,
        "share_parameter": True,
        "centralized_critic": True
    }

actor_configs = {
        "num_cells":256,
        "depth":2,
        "share_parameter":False
    }

MAX_QUEUE_LENGTH = 5 #5 times capacity
PENALTY = -5
ALPHA = 1.0
BETA = 1000.0


### Experiment 1

- Single slice
- Constant Demand = 0.5
- (lambda, rho) = (0.5,0.5), (0.1,0.9), (0.9,0.1)

In [ ]:
n_agent = 1
test_demand = 0.5
latency_pref = [0.5, 0.1, 0.9]
energy_pref = [0.5, 0.9, 0.1]


In [ ]:
for idx, (lambda_, rho_) in enumerate(zip(latency_pref, energy_pref)):
    print(f'idx={idx}, lambda = {lambda_}, rho = {rho_}')
    
    log_path = os.path.join(LOG_PATH,f"1.{idx}")
    
    train_env = NetworkEnvV8(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path= os.path.join(log_path,'train'),
        test_demand=test_demand,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[lambda_],
        energy_preference=[rho_]
    )

    test_env = NetworkEnvV8(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path=os.path.join(log_path,'test'),
        test_demand=test_demand,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[lambda_],
        energy_preference=[rho_]
    )

    trainer = MADDPGTrainer(
        environment=train_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=n_iter,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        save_path=log_path,
        seed = 0
    )

    tester = MADDPGTester(
        environment=test_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=10,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        load_path=log_path
    )
    
    trainer.train()
    tester.test()
    train_env.save_statistics()
    test_env.save_statistics()

### Experiment 2

- Single slice
- Constant Demand = 0.3,0.5,0.7
- (lambda, rho) = (0.5,0.5)

In [ ]:
n_agent = 1
test_demand = [0.3,0.5,0.7]
latency_pref = 0.5
energy_pref = 0.5

In [ ]:
for idx, demand_ in enumerate(test_demand):
    print(f'idx={idx}, demand = {demand_}')
    
    log_path = os.path.join(LOG_PATH,f"2.{idx}")
    
    train_env = NetworkEnvV8(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path= os.path.join(log_path,'train'),
        test_demand=demand_,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[latency_pref],
        energy_preference=[energy_pref]
    )

    test_env = NetworkEnvV8(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path=os.path.join(log_path,'test'),
        test_demand=demand_,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[latency_pref],
        energy_preference=[energy_pref]
    )

    trainer = MADDPGTrainer(
        environment=train_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=n_iter,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        save_path=log_path,
        seed = 0
    )

    tester = MADDPGTester(
        environment=test_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=10,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        load_path=log_path
    )
    
    trainer.train()
    tester.test()
    train_env.save_statistics()
    test_env.save_statistics()

### Experiment 3

- Single slice
- Uniform demand [0.3, 0.7]
- (lambda, rho) = (0.5,0.5)

In [ ]:
lambda_ = 0.5
rho_ = 0.5

traffic_path_train = os.path.join(DATA,"train")
traffic_path_test = os.path.join(DATA,"test")

In [ ]:
log_path = os.path.join(LOG_PATH,f"3")

train_env = NetworkEnvV8(
    n_slices=n_agent,
    resource_path=RESOURCE_PATH,
    traffic_path=traffic_path_train,
    log_path= os.path.join(log_path,'train'),
    test_demand= None,
    max_queue_length=MAX_QUEUE_LENGTH,
    penalty_reward=PENALTY,
    latency_preference=[lambda_],
    energy_preference=[rho_]
)

test_env = NetworkEnvV8(
    n_slices=n_agent,
    resource_path=RESOURCE_PATH,
    traffic_path=traffic_path_test,
    log_path=os.path.join(log_path,'test'),
    test_demand=None,
    max_queue_length=MAX_QUEUE_LENGTH,
    penalty_reward=PENALTY,
    latency_preference=[lambda_],
    energy_preference=[rho_]
)

trainer = MADDPGTrainer(
    environment=train_env,
    n_agent=n_agent,
    frames_per_batch=frames_per_batch,
    n_iter=n_iter,
    min_replay_size=min_replay_size,
    memory_size=memory_size,
    n_optimizer_steps= n_optimizer_steps,
    train_batch_size=train_batch_size,
    actor_lr=actor_lr,
    critic_lr=critic_lr,
    max_grad_norm=max_grad_norm,
    polyak_tau=polyak_tau,
    gamma=gamma,
    critic_configs=critic_configs,
    actor_configs=actor_configs,
    save_path=log_path,
    seed = 0
)

tester = MADDPGTester(
    environment=test_env,
    n_agent=n_agent,
    frames_per_batch=frames_per_batch,
    n_iter=10,
    min_replay_size=min_replay_size,
    memory_size=memory_size,
    n_optimizer_steps= n_optimizer_steps,
    train_batch_size=train_batch_size,
    actor_lr=actor_lr,
    critic_lr=critic_lr,
    max_grad_norm=max_grad_norm,
    polyak_tau=polyak_tau,
    gamma=gamma,
    critic_configs=critic_configs,
    actor_configs=actor_configs,
    load_path=log_path
)

trainer.train()
tester.test()
train_env.save_statistics()
test_env.save_statistics()

### Experiment 4

- 3 slices
- Uniform demand [0.3, 0.7]
- latency_pref = [0.5, 0.1, 0.9]
- energy_pref = [0.5, 0.9, 0.1]

In [ ]:
lambda_ = [0.5,0.1,0.9]
rho_ = [0.5, 0.1, 0.9]

n_agent = 3

traffic_path_train = os.path.join(DATA,"train")
traffic_path_test = os.path.join(DATA,"test")

frames_per_batch = 1000
n_iter = 100
min_replay_size = 10000
memory_size = 300000
n_optimizer_steps = 20
train_batch_size = 512
actor_lr = 1e-5
critic_lr = 1e-5
max_grad_norm = 0.5
gamma = 0.95
polyak_tau = 0.001

PENALTY = -1


In [ ]:
log_path = os.path.join(LOG_PATH,f"4")

train_env = NetworkEnvV8(
    n_slices=n_agent,
    resource_path=RESOURCE_PATH,
    traffic_path=traffic_path_train,
    log_path= os.path.join(log_path,'train'),
    test_demand= None,
    max_queue_length=MAX_QUEUE_LENGTH,
    penalty_reward=PENALTY,
    latency_preference=lambda_,
    energy_preference=rho_
)

test_env = NetworkEnvV8(
    n_slices=n_agent,
    resource_path=RESOURCE_PATH,
    traffic_path=traffic_path_test,
    log_path=os.path.join(log_path,'test'),
    test_demand=None,
    max_queue_length=MAX_QUEUE_LENGTH,
    penalty_reward=PENALTY,
    latency_preference=lambda_,
    energy_preference=rho_
)

for resource in train_env.resources.values():
    resource.capacity = resource.capacity * n_agent 
    resource.avalable_capacity = resource.capacity

for resource in test_env.resources.values():
    resource.capacity = resource.capacity * n_agent 
    resource.avalable_capacity = resource.capacity


trainer = MADDPGTrainer(
    environment=train_env,
    n_agent=n_agent,
    frames_per_batch=frames_per_batch,
    n_iter=n_iter,
    min_replay_size=min_replay_size,
    memory_size=memory_size,
    n_optimizer_steps= n_optimizer_steps,
    train_batch_size=train_batch_size,
    actor_lr=actor_lr,
    critic_lr=critic_lr,
    max_grad_norm=max_grad_norm,
    polyak_tau=polyak_tau,
    gamma=gamma,
    critic_configs=critic_configs,
    actor_configs=actor_configs,
    save_path=log_path,
    seed = 0,
    random_frames=10000,
    noise_sigma=0.05
)

tester = MADDPGTester(
    environment=test_env,
    n_agent=n_agent,
    frames_per_batch=frames_per_batch,
    n_iter=10,
    min_replay_size=min_replay_size,
    memory_size=memory_size,
    n_optimizer_steps= n_optimizer_steps,
    train_batch_size=train_batch_size,
    actor_lr=actor_lr,
    critic_lr=critic_lr,
    max_grad_norm=max_grad_norm,
    polyak_tau=polyak_tau,
    gamma=gamma,
    critic_configs=critic_configs,
    actor_configs=actor_configs,
    load_path=log_path
)

trainer.train()
tester.test()
train_env.save_statistics()
test_env.save_statistics()